In [0]:
import json
import logging

from typing import List, Dict, Any, Optional

from pyspark.sql            import DataFrame, SparkSession
from pyspark.sql            import functions as F
from pyspark.sql.types      import TimestampType
from delta.tables           import DeltaTable

In [0]:


def get_logger(notebook_name: str) -> logging.Logger:
    
    logger = logging.getLogger(f"silver.{notebook_name}")
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%Y-%m-%dT%H:%M:%SZ"
        ))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger



In [0]:

 
def get_watermark(
    spark          : SparkSession,
    source_table   : str,
    watermark_path : str,
    logger         : logging.Logger,
) -> Optional[object]:
    
    table_exists = DeltaTable.isDeltaTable(spark, watermark_path)
 
    if not table_exists:
        logger.info(f"  Watermark table does not exist yet — first run for all tables")
        return None
 
    wm_df = (
        spark.read.format("delta").load(watermark_path)
        .filter(F.col("source_table") == source_table)
    )
 
    if wm_df.count() == 0:
        logger.info(f"  No watermark found for '{source_table}' — will read all Bronze rows")
        return None
 
    last_ts = wm_df.collect()[0]["last_processed_ts"]
    logger.info(f"  Watermark for '{source_table}': {last_ts}")
    return last_ts
 
 
def update_watermark(
    spark          : SparkSession,
    source_table   : str,
    new_ts         : object,
    watermark_path : str,
    run_ts         : object,
    logger         : logging.Logger,
) -> None:
    
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
 
    logger.info(f"  Updating watermark: '{source_table}' → {new_ts}")
 
    # Build a single-row DataFrame for the new watermark value
    new_wm_df = spark.createDataFrame(
        [(source_table, new_ts, run_ts)],
        schema=StructType([
            StructField("source_table",      StringType(),   False),
            StructField("last_processed_ts", TimestampType(), False),
            StructField("updated_at",        TimestampType(), False),
        ])
    )
 
    table_exists = DeltaTable.isDeltaTable(spark, watermark_path)
 
    if not table_exists:
        
        (
            new_wm_df.write
            .format("delta")
            .mode("overwrite")
            .save(watermark_path)
        )
        logger.info(f"  ✓ Watermark table created at: {watermark_path}")
        return
 
    
    wm_table = DeltaTable.forPath(spark, watermark_path)
    (
        wm_table.alias("existing")
        .merge(
            new_wm_df.alias("new"),
            "existing.source_table = new.source_table"
        )
        .whenMatchedUpdate(set={
            "last_processed_ts": "new.last_processed_ts",
            "updated_at"       : "new.updated_at",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
 
    logger.info(f"  ✓ Watermark updated for '{source_table}'")
 
 

In [0]:


def read_bronze_latest(
    spark        : SparkSession,
    bronze_path  : str,
    logger       : logging.Logger,
) -> DataFrame:
    
    logger.info(f"  Reading Bronze: {bronze_path}")

    
    bronze_df = spark.read.format("delta").load(bronze_path)

    
    max_ts_row = bronze_df.agg(F.max("bronze_loaded_at").alias("max_ts")).collect()[0]
    max_ts     = max_ts_row["max_ts"]

    logger.info(f"  Latest bronze_loaded_at: {max_ts}")

    
    latest_df = bronze_df.filter(F.col("bronze_loaded_at") == max_ts)

    count = latest_df.count()
    logger.info(f"  Latest batch row count: {count:,}")

    return latest_df

def read_bronze_incremental(
    spark          : SparkSession,
    bronze_path    : str,
    source_table   : str,
    watermark_path : str,
    logger         : logging.Logger,
) -> tuple:
    
    logger.info(f"  Incremental Bronze read: {bronze_path}")
    
 
    
    last_processed_ts = get_watermark(spark, source_table, watermark_path, logger)
    print(last_processed_ts)
 
   
    bronze_df = spark.read.format("delta").load(bronze_path)
 
    if last_processed_ts is None:
        
        logger.info(f"  First run: reading all Bronze rows")
        new_df = bronze_df
    else:
        new_df = bronze_df.filter(F.col("bronze_loaded_at") > last_processed_ts)
 
    count = new_df.count()
    logger.info(f"  New rows to process: {count:,}")
 
    if count == 0:
        logger.info(f"  No new Bronze data — Silver is already up to date")
        return new_df, None
 
    
    max_ts_row = new_df.agg(F.max("bronze_loaded_at").alias("max_ts")).collect()[0]
    max_ts     = max_ts_row["max_ts"]
    logger.info(f"  Max bronze_loaded_at in this batch: {max_ts}")
 
    return new_df, max_ts

In [0]:


def delta_merge(
    spark       : SparkSession,
    new_df      : DataFrame,
    table_path  : str,
    merge_keys  : List[str],
    logger      : logging.Logger,
) -> Dict[str, Any]:
    
    merge_condition = " AND ".join(
        [f"existing.{col} = new.{col}" for col in merge_keys]
    )
    logger.info(f"  MERGE into: {table_path}")
    logger.info(f"  Merge condition: {merge_condition}")

    
    table_exists = DeltaTable.isDeltaTable(spark, table_path)

    if not table_exists:
        
        logger.info("  Table does not exist yet — creating on first write")
        (
            new_df
            .write
            .format("delta")
            .mode("overwrite")   
            .option("mergeSchema", "true")
            .save(table_path)
        )
        rows_after = new_df.count()
        logger.info(f"  ✓ Table created | rows: {rows_after:,}")
        return {"rows_before": 0, "rows_after": rows_after, "rows_inserted": rows_after}

    
    delta_table = DeltaTable.forPath(spark, table_path)

    
    rows_before = delta_table.toDF().count()

    (
        delta_table.alias("existing")
        .merge(
            new_df.alias("new"),
            merge_condition
        )
        
        .whenNotMatchedInsertAll()
        .execute()
    )

    rows_after   = delta_table.toDF().count()
    rows_inserted = rows_after - rows_before

    logger.info(f"  ✓ MERGE complete | before: {rows_before:,} | after: {rows_after:,} | inserted: {rows_inserted:,}")
    return {
        "rows_before"  : rows_before,
        "rows_after"   : rows_after,
        "rows_inserted": rows_inserted,
    }



In [0]:


def validate_drop_rate(
    rows_before  : int,
    rows_after   : int,
    max_fraction : float,
    table_name   : str,
    logger       : logging.Logger,
) -> None:
    
    if rows_before == 0:
        logger.warning(f"  [{table_name}] Input batch has 0 rows — nothing to validate")
        return

    dropped  = rows_before - rows_after
    fraction = dropped / rows_before

    logger.info(f"  [{table_name}] Quality filter: {dropped:,} rows dropped ({fraction:.1%} of {rows_before:,})")

    if fraction > max_fraction:
        msg = (
            f"Data quality check FAILED for {table_name}: "
            f"{fraction:.1%} of rows dropped (max allowed: {max_fraction:.1%}). "
            f"rows_before={rows_before:,}, rows_after={rows_after:,}. "
            f"Investigate Bronze source data before proceeding."
        )
        logger.error(f"  {msg}")
        raise ValueError(msg)

In [0]:


def write_run_log(
    summary    : Dict[str, Any],
    log_path   : str,
    logger     : logging.Logger,
) -> None:
    
    try:
        log_content = json.dumps(summary, indent=2, default=str)
        dbutils.fs.put(log_path, log_content, overwrite=True)
        logger.info(f"  ✓ Run log written → {log_path}")
    except Exception as e:
        logger.warning(f"  ⚠ Failed to write run log (non-fatal): {e}")




def get_silver_timestamp(run_ts) -> "Column":
    return F.lit(run_ts.isoformat()).cast(TimestampType())



In [0]:


def log_schema(df: DataFrame, label: str, logger: logging.Logger) -> None:
    """Print the DataFrame schema to the logger at DEBUG level."""
    schema_str = df.schema.simpleString()
    logger.debug(f"  Schema [{label}]: {schema_str}")




def assert_required_columns(
    df              : DataFrame,
    required_cols   : List[str],
    table_name      : str,
    logger          : logging.Logger,
) -> None:
    
    existing = set(df.columns)
    missing  = set(required_cols) - existing

    if missing:
        msg = (
            f"Missing required columns in {table_name}: {sorted(missing)}. "
            f"Available columns: {sorted(existing)}. "
            f"Check if the CoinGecko API response format has changed."
        )
        logger.error(f"  {msg}")
        raise ValueError(msg)

    logger.info(f"  ✓ All {len(required_cols)} required columns present in {table_name}")